# Bank Customer Churn Prediction Using Random Forest and XGBoost

**Roll Numbers:** 2520030248, 2520030273, 2520030270
**Department:** Computer Science Engineering, KLH University

This notebook builds a machine learning system that predicts whether a bank customer
will churn (leave the bank), using **Random Forest** and **XGBoost** classifiers.
We compare both models on Accuracy, Precision, Recall, F1-score, and ROC-AUC, as
described in the project abstract.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, roc_curve)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (7, 5)

## 2. Load the Dataset

We use `bank_churn.csv`, which contains 10,000 customer records with the exact
features described in the abstract: credit score, geography, gender, age, tenure,
balance, number of products, credit card status, active membership, estimated
salary, and the churn label.

In [ ]:
df = pd.read_csv('bank_churn.csv')
print("Dataset shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## 3. Exploratory Data Analysis (EDA)

Before modeling, we look at the churn distribution and how key features relate to churn.

In [ ]:
plt.figure()
sns.countplot(x='churn', data=df, palette=['#2E86AB', '#E63946'])
plt.title('Churn Distribution (0 = Stayed, 1 = Churned)')
plt.xlabel('Churn')
plt.ylabel('Number of Customers')
plt.show()

print(df['churn'].value_counts(normalize=True) * 100)

In [ ]:
plt.figure()
sns.countplot(x='country', hue='churn', data=df, palette=['#2E86AB', '#E63946'])
plt.title('Churn Count by Country')
plt.show()

In [ ]:
plt.figure()
sns.histplot(data=df, x='age', hue='churn', bins=30, kde=True, palette=['#2E86AB', '#E63946'])
plt.title('Age Distribution by Churn Status')
plt.show()

In [ ]:
plt.figure(figsize=(9,7))
numeric_df = df.drop(columns=['customer_id'])
numeric_df['gender_num'] = numeric_df['gender'].map({'Male': 1, 'Female': 0})
corr = numeric_df.drop(columns=['gender', 'country']).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## 4. Data Preprocessing

Steps:
1. Drop `customer_id` — it's just an identifier, not predictive.
2. Encode `gender` as 0/1 using Label Encoding.
3. One-hot encode `country` (France is the baseline).
4. Split into features (X) and target (y).
5. Train/test split (80/20), stratified so both sets keep the same churn ratio.
6. Scale features with `StandardScaler`.

In [ ]:
data = df.drop(columns=['customer_id']).copy()

le_gender = LabelEncoder()
data['gender'] = le_gender.fit_transform(data['gender'])  # Female=0, Male=1

data = pd.get_dummies(data, columns=['country'], drop_first=True)

X = data.drop(columns=['churn'])
y = data['churn']

print("Final feature columns:", list(X.columns))
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train size:", X_train.shape, " Test size:", X_test.shape)

## 5. Model 1 — Random Forest

Random Forest builds many decision trees on random subsets of data/features and
averages their votes. `class_weight='balanced'` helps because churn (~20%) is the
minority class.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=10, random_state=42, class_weight='balanced'
)
rf.fit(X_train_scaled, y_train)

rf_pred = rf.predict(X_test_scaled)
rf_proba = rf.predict_proba(X_test_scaled)[:, 1]

## 6. Model 2 — XGBoost

XGBoost builds trees sequentially, where each new tree corrects the errors of the
previous ones (gradient boosting). `scale_pos_weight` compensates for class imbalance.

In [ ]:
pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    eval_metric='logloss', random_state=42,
    scale_pos_weight=pos_weight
)
xgb.fit(X_train_scaled, y_train)

xgb_pred = xgb.predict(X_test_scaled)
xgb_proba = xgb.predict_proba(X_test_scaled)[:, 1]

## 7. Model Evaluation

We compare both models using Accuracy, Precision, Recall, F1-score, and ROC-AUC —
exactly the metrics named in the abstract.

In [ ]:
def get_metrics(name, y_true, y_pred, y_proba):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_proba)
    }

results = pd.DataFrame([
    get_metrics('Random Forest', y_test, rf_pred, rf_proba),
    get_metrics('XGBoost', y_test, xgb_pred, xgb_proba)
]).set_index('Model')

results.round(4)

In [ ]:
results.round(3).plot(kind='bar', figsize=(10,6), colormap='viridis')
plt.title('Random Forest vs XGBoost — Metric Comparison')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, pred) in zip(axes, [('Random Forest', rf_pred), ('XGBoost', xgb_pred)]):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Stayed', 'Churned'], yticklabels=['Stayed', 'Churned'])
    ax.set_title(f'{name} — Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

### ROC Curves

In [ ]:
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_proba)

plt.figure()
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {roc_auc_score(y_test, rf_proba):.3f})")
plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC = {roc_auc_score(y_test, xgb_proba):.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()

### Detailed Classification Reports

In [ ]:
print("Random Forest:\n", classification_report(y_test, rf_pred, target_names=['Stayed','Churned']))
print("XGBoost:\n", classification_report(y_test, xgb_pred, target_names=['Stayed','Churned']))

## 8. Feature Importance

Which features matter most for predicting churn? This helps the bank understand
*why* customers leave, not just *who* is likely to leave.

In [ ]:
rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
xgb_importance = pd.Series(xgb.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
rf_importance.plot(kind='barh', ax=axes[0], color='#2E86AB')
axes[0].invert_yaxis()
axes[0].set_title('Random Forest — Feature Importance')

xgb_importance.plot(kind='barh', ax=axes[1], color='#E63946')
axes[1].invert_yaxis()
axes[1].set_title('XGBoost — Feature Importance')

plt.tight_layout()
plt.show()

## 9. Conclusion

- Both **Random Forest** and **XGBoost** were trained on 10,000 bank customer
  records using the features described in the abstract (age, gender, geography,
  credit score, tenure, balance, number of products, credit card status, active
  membership, estimated salary).
- Model performance was evaluated using Accuracy, Precision, Recall, F1-score, and
  ROC-AUC (see the comparison table/chart in Section 7).
- **Age**, **number of products**, and **account balance** consistently show up as
  the strongest predictors of churn across both models.
- The better-performing model (by ROC-AUC / F1-score, whichever the project
  prioritizes) can be deployed by the bank to flag high-risk customers early, so
  retention teams can act with personalized offers or better engagement before
  the customer actually leaves.

**Possible extensions:** hyperparameter tuning (GridSearchCV/Optuna), SMOTE for
class imbalance, SHAP values for explainability, and deploying the final model as
a simple Flask/Streamlit app.

## 10. Which specific customers are most likely to leave?

The metrics above tell us how *good* the models are overall. But in practice, a
bank wants an actual list: "here are the top 20 customers most at risk right now,
go call them." This is done using `predict_proba()`, which gives a **risk score
between 0 and 1** for every customer (0 = definitely stays, 1 = definitely leaves)
instead of just a plain yes/no.

In [ ]:
# Build a results table for the test customers (the 2000 unseen ones)
risk_table = X_test.copy()
risk_table['Actual_Churn'] = y_test.values
risk_table['RF_Risk_Score'] = rf_proba          # Random Forest's risk score (0 to 1)
risk_table['XGB_Risk_Score'] = xgb_proba        # XGBoost's risk score (0 to 1)

# Sort by Random Forest's risk score, highest risk first
# Show ALL customers the model predicts will churn (not just top N)
all_predicted_churners = risk_table[risk_table['RF_Risk_Score'] >= 0.5].sort_values('RF_Risk_Score', ascending=False)

print(f"The model predicts {len(all_predicted_churners)} out of {len(risk_table)} test customers will churn.")
all_predicted_churners[['age', 'balance', 'products_number', 'active_member',
                         'Actual_Churn', 'RF_Risk_Score', 'XGB_Risk_Score']]

**How to read this table:** each row is a real customer from our test set.
`RF_Risk_Score` close to 1.0 means the model is very confident this customer will
leave. `Actual_Churn` shows whether they actually left (1) or stayed (0) — so you
can see for yourself how often the model's top-risk list matches reality.

In a real deployment, the bank would take this exact table, sort by risk score,
and hand the top rows to the retention team to reach out to those customers with
offers before they leave — that's the entire business value of this project.

In [ ]:
   # How many of the FULL predicted-churner list (all 435) actually did churn?
   full_hit_rate = all_predicted_churners['Actual_Churn'].sum() / len(all_predicted_churners) * 100
   print(f"Of all {len(all_predicted_churners)} customers the model flagged as likely to churn, "
         f"{all_predicted_churners['Actual_Churn'].sum()} out of {len(all_predicted_churners)} "
         f"({full_hit_rate:.0f}%) actually churned.")